## Get the Name of The Company 

In [76]:

ticker = "CRCL"

In [83]:
from tavily import TavilyClient

def get_company_name_by_ticker(ticker: str) -> str:
    """
    Get company name from ticker symbol using Tavily search.
    
    Args:
        ticker (str): Stock ticker symbol (e.g., "BBW", "AAPL", "TSLA")
    
    Returns:
        str: Company name
    """
    client = TavilyClient("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")
    
    # Enhanced query to get company name only
    query = f"what is the company name for stock ticker {ticker.upper()}, return only the company name"
    
    try:
        response = client.search(
            query=query,
            include_answer="advanced",
            search_depth="advanced"
        )
        
        # Extract company name from response
        if response and 'answer' in response:
            company_name = response['answer']
            # Clean up the response to get just the company name
            company_name = company_name.strip()
            # Remove any extra text after the company name
            if '\n' in company_name:
                company_name = company_name.split('\n')[0]
            if '.' in company_name and len(company_name.split('.')[0]) < 20:
                company_name = company_name.split('.')[0]
            
            return company_name
        else:
            return f"Company for {ticker.upper()}"
            
    except Exception as e:
        print(f"Error getting company name for {ticker}: {e}")
        return f"Company for {ticker.upper()}"


In [84]:

company_name = get_company_name_by_ticker(ticker)
print(f"Company: {company_name}")


Company: Circle Internet Group, Inc.


## Asset Relative Search

In [85]:
def asset_relative_check(company_name):
    """
    Check what product/service/asset a company is relative to.
    
    Args:
        company_name (str): Name of the company to check
        
    Returns:
        str: The answer from Tavily response
    """
    from tavily import TavilyClient
    
    client = TavilyClient("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")
    
    response = client.search(
        query=f"What is the product /service/asset relative that {company_name} lies on? Market Expect on? Currently? One key sentence that capture the company's value (10 words)",
        include_answer="advanced",
        search_depth="advanced"
    )
    
    return response.get("answer", "No answer found")

In [86]:
asset_relative = asset_relative_check(company_name)

## Sector Landscape and Competitor Check

### Chain of Thought

In [87]:
import sys
import os
from pathlib import Path

# For Jupyter notebook - use current working directory
current_dir = Path.cwd()
parent_dir = current_dir.parent
sys.path.insert(0, str(parent_dir))

from pydantic import BaseModel, Field, ValidationError
from shared_clients import shared_clients

class SectorAnalysisQueries(BaseModel):
    """Schema for sector analysis queries generated by LLM."""
    trend_query: str = Field(
        description="Query about trends in policy, tech, consumer market in US and Global (max 350 characters)"
    )
    competitor_query: str = Field(
        description="Query about competitor landscape, market distribution percentages (max 350 characters)"
    )

async def generate_sector_queries(asset_relative: str, company: str) -> SectorAnalysisQueries:
    """
    Generate sector analysis queries using LLM with structured output.
    
    Args:
        asset_relative (str): The asset/product/service the company is relative to
        company (str): Company name to include in competitor analysis
        
    Returns:
        SectorAnalysisQueries: Structured queries for sector analysis
    """
    try:
        # Initialize shared clients
        await shared_clients.initialize()
        
        # Create prompt for LLM with character limit
        prompt = f"""
        Based on the asset/product/service: "{asset_relative}" and company: "{company}",
        generate two specific analysis queries (each must be under 350 characters):
        
        1. Trend Query: About trends in policy, tech, consumer market in US and Global for {asset_relative}, this query is try to understand The Company' industrial might impact itself. 
        2. Competitor Query: About competitor landscape, market distribution percentages, including {company}, this query is tring to understand how the company is doing in the market.
        
        IMPORTANT: Each query must be concise and under 350 characters. Make them specific and actionable.
        """
        
        # Get LLM call agent from shared clients
        llm_call_agent = shared_clients.get_llm_agent()
        
        # Get structured LLM
        structured_llm = llm_call_agent.get_structured_llm(SectorAnalysisQueries)
        
        # Call LLM with structured output
        result = await structured_llm.ainvoke(prompt)
        
        # Validate character limits
        if len(result.trend_query) > 350:
            result.trend_query = result.trend_query[:347] + "..."
        if len(result.competitor_query) > 350:
            result.competitor_query = result.competitor_query[:347] + "..."
        
        return result
        
    except Exception as e:
        print(f"❌ Error generating sector queries: {e}")
        # Return default queries if LLM fails (under 350 chars)
        return SectorAnalysisQueries(
            trend_query=f"What are the trends in {asset_relative} (policy, tech, consumer market in US and Global)?",
            competitor_query=f"What is the competitor landscape and market distribution for {asset_relative}, including {company}?"
        )

# Generate queries using LLM Chain of Thought
queries = await generate_sector_queries(asset_relative, company_name)
Sector_trend_query = queries.trend_query
Sectorcompetitor_query = queries.competitor_query

print("🎯 Generated Sector Analysis Queries:")
print(f"�� Trend Query ({len(Sector_trend_query)} chars): {Sector_trend_query}")
print(f"🏆 Competitor Query ({len(Sectorcompetitor_query)} chars): {Sectorcompetitor_query}")

✅ Shared clients already initialized
2025-09-12 19:48:47,640 - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
🎯 Generated Sector Analysis Queries:
�� Trend Query (235 chars): What are the current US and global regulatory trends for stablecoin adoption and blockchain payments infrastructure? How are CBDC developments and tokenized asset markets impacting Circle's USDC growth prospects and market positioning?
🏆 Competitor Query (248 chars): What is the current stablecoin market share distribution between USDC, USDT, and other major stablecoins? How does Circle's market cap and revenue compare to Tether and other blockchain payment infrastructure competitors in the digital asset space?


#### Calling Internet Search Again 

In [91]:
import asyncio
from tavily import TavilyClient
from datetime import datetime
import json

async def concurrent_tavily_search(trend_query: str, competitor_query: str):
    """
    Run both Tavily searches concurrently.
    
    Args:
        trend_query (str): Query about trends
        competitor_query (str): Query about competitors
        
    Returns:
        dict: Results from both searches
    """
    client = TavilyClient("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")
    
    async def search_trends():
        """Search for trends"""
        try:
            print(f"🔍 Searching trends: {trend_query}")
            response = client.search(
                query=trend_query,
                include_answer="advanced",
                search_depth="advanced",
                max_results=10
            )
            print(f"✅ Trends search completed")
            
            # Extract all URLs from results
            all_urls = [result.get('url', '') for result in response.get('results', []) if result.get('url')]
            
            return {
                "type": "trends",
                "query": trend_query,
                "answer": response.get('answer', ''),
                "urls": all_urls,
                "status": "success"
            }
        except Exception as e:
            print(f"❌ Trends search failed: {e}")
            return {
                "type": "trends",
                "query": trend_query,
                "answer": "",
                "urls": [],
                "error": str(e),
                "status": "failed"
            }
    
    async def search_competitors():
        """Search for competitors"""
        try:
            print(f"🔍 Searching competitors: {competitor_query}")
            response = client.search(
                query=competitor_query,
                include_answer="advanced",
                search_depth="advanced",
                max_results=10
            )
            print(f"✅ Competitors search completed")
            
            # Extract all URLs from results
            all_urls = [result.get('url', '') for result in response.get('results', []) if result.get('url')]
            
            return {
                "type": "competitors",
                "query": competitor_query,
                "answer": response.get('answer', ''),
                "urls": all_urls,
                "status": "success"
            }
        except Exception as e:
            print(f"❌ Competitors search failed: {e}")
            return {
                "type": "competitors",
                "query": competitor_query,
                "answer": "",
                "urls": [],
                "error": str(e),
                "status": "failed"
            }
    
    print("🚀 Starting concurrent Tavily searches...")
    print("=" * 60)
    
    # Run both searches concurrently
    trend_task = asyncio.create_task(search_trends())
    competitor_task = asyncio.create_task(search_competitors())
    
    # Wait for both to complete
    trend_result, competitor_result = await asyncio.gather(trend_task, competitor_task)
    
    print("\n" + "=" * 60)
    print("🎉 Both searches completed!")
    print("=" * 60)
    
    return {
        "trends": trend_result,
        "competitors": competitor_result,
        "timestamp": datetime.now().isoformat()
    }

# Usage with your existing variables
search_results = await concurrent_tavily_search(Sector_trend_query, Sectorcompetitor_query)

# Structure the outputs as requested
answer_collection = {
    "sector_trend": search_results['trends']['answer'],
    "company_competitor_landscape": search_results['competitors']['answer']
}

url_collection = {
    "sector_trend": search_results['trends']['urls'],
    "company_competitor_landscape": search_results['competitors']['urls']
}

# Display structured results
print(f"\n📊 Answer Collection JSON:")
print(json.dumps(answer_collection, indent=2))

print(f"\n🔗 URL Collection JSON:")
print(json.dumps(url_collection, indent=2))

print(f"\n📈 Summary:")
print(f"   Trends status: {search_results['trends']['status']}")
print(f"   Competitors status: {search_results['competitors']['status']}")
print(f"   Total URLs found: {len(search_results['trends']['urls']) + len(search_results['competitors']['urls'])}")

🚀 Starting concurrent Tavily searches...
🔍 Searching trends: What are the current US and global regulatory trends for stablecoin adoption and blockchain payments infrastructure? How are CBDC developments and tokenized asset markets impacting Circle's USDC growth prospects and market positioning?
✅ Trends search completed
🔍 Searching competitors: What is the current stablecoin market share distribution between USDC, USDT, and other major stablecoins? How does Circle's market cap and revenue compare to Tether and other blockchain payment infrastructure competitors in the digital asset space?
✅ Competitors search completed

🎉 Both searches completed!

📊 Answer Collection JSON:
{
  "sector_trend": "The regulatory landscape for stablecoins is experiencing unprecedented momentum globally, with the US recently passing the GENIUS Act which establishes a federal regulatory framework for payment stablecoins and marks a historic shift toward regulatory clarity. North America shows 88% favorable s

## Storage py 

In [98]:
asset_relative

"Circle Internet Group, Inc. operates as a peer-to-peer payments technology company and platform infrastructure provider for stablecoin and blockchain applications, with its primary product being USDC, the second largest stablecoin worldwide that maintains a stable value pegged to the U.S. dollar. The company trades on the NYSE under ticker CRCL following its June 2025 IPO that raised $1.1 billion and valued the company at $6.9 billion, with shares more than doubling in their debut. Circle's business model centers on issuing and managing USDC while providing developer services, payments tools, and treasury products that enable companies to move currencies at internet speed with lower fees and faster settlement than traditional systems. The market expects continued growth in the digital asset and stablecoin infrastructure space, with Circle positioned to benefit from expanding adoption of blockchain-based financial services and tokenized asset markets. Currently, the company reports rev

In [99]:
answer_collection = {
    "sector_trend": search_results['trends']['answer'],
    "company_competitor_landscape": search_results['competitors']['answer']
}

answer_collection

{'sector_trend': "The regulatory landscape for stablecoins is experiencing unprecedented momentum globally, with the US recently passing the GENIUS Act which establishes a federal regulatory framework for payment stablecoins and marks a historic shift toward regulatory clarity. North America shows 88% favorable sentiment toward upcoming stablecoin regulations, while Europe has already implemented the MiCA framework with Circle becoming the first major issuer to comply with these rules for both USDC and EURC. The Trump administration has created significant regulatory momentum by discontinuing US CBDC exploration through executive order, positioning stablecoins as the preferred digital dollar solution. USDC has demonstrated exceptional growth with a 78% year-over-year increase in circulation, monthly transaction volumes reaching $1 trillion in November 2024, and total circulation exceeding $18 trillion, outpacing all other major global stablecoins. Circle's market positioning has been s

In [100]:
url_collection = {
    "sector_trend": search_results['trends']['urls'],
    "company_competitor_landscape": search_results['competitors']['urls']
}

url_collection 

{'sector_trend': ['https://www.fireblocks.com/report/state-of-stablecoins/',
  'https://www.circle.com/reports/state-of-the-usdc-economy',
  'https://s206.q4cdn.com/265218871/files/doc_financials/2025/q2/Q2-2025-Earnings-Press-Release.pdf',
  'https://www.chainalysis.com/blog/2025-global-crypto-adoption-index/',
  'https://www.circle.com/reports/state-of-the-usdc-economy/usdc-the-worlds-digital-dollar',
  'https://markets.financialcontent.com/wral/article/marketminute-2025-9-9-stablecoin-revolution-global-regulations-reshape-digital-finance-landscape',
  'https://economics.td.com/us-stablecoins-enter-the-mainstream',
  'https://privatebank.barclays.com/insights/stablecoins-the-new-generation-of-financial-infrastructure-07-2025/',
  'https://media-publications.bcg.com/Stablecoins-five-killer-tests-to-gauge-their-potential.pdf',
  'https://www.fxcintel.com/research/reports/ct-state-of-stablecoins-cross-border-payments-2025'],
 'company_competitor_landscape': ['https://medium.com/@monolit